# Primary MERFISH biological figures

This notebook generates the primary MERFISH biological figures:

- **Figure 2A:** spatial examples illustrating within-batch age-label permutation.
- **Figure 3A:** representative anterior and posterior cell-type and broad-region maps.
- **Figure 3B:** recurrent-gene effect-size/significance dot plot.
- **Figure 3C:** split violin examples for raw expression.
- **Figure 6C:** representative spatial expression of `Cpe`.

The notebook preserves the original analysis logic while consolidating repeated setup and plotting operations into reusable functions.


## 0. Setup

Update the paths below for the local installation. The anterior and posterior Vizgen directories correspond to the representative matched young/old section pairs used in the manuscript figures.


In [ ]:
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import squidpy as sq
import matplotlib.pyplot as plt

from matplotlib.collections import PolyCollection
from matplotlib.colors import to_rgb
from matplotlib.lines import Line2D

import kimlabspatial.preprocessing as pp

from scale_aware_st import RepositoryConfig, index_result_tables, load_pooled_strata


In [ ]:
# Repository, output, and deposited-data paths
config = RepositoryConfig.from_env()
ALDEX_RESULT_MODE = "published"  # choose "published" or "recompute"
project_dir = config.root
deg_dir = config.results_dir / "DEG"

aldex_results_file = deg_dir / "aldex_ct_inftss_results.xlsx"
gene_pivot_file = deg_dir / "ct_region_inftss_deg_all_pivot_signed.xlsx"
adata_file = config.data_dir / "primary_merfish" / "analysis_objects" / "adata_ct_centered_dec25.hdf5"
ontology_file = config.resources_dir / "allen_ccf" / "CCFv3OntologyStructure_u16.xlsx"
glia_adata_file = config.data_dir / "primary_merfish" / "analysis_objects" / "adata_glia_aldex_pp.h5ad"

spatial_section_dir = config.data_dir / "primary_merfish" / "spatial_sections"
anterior_young_dir = spatial_section_dir / "young_anterior_section_example_1"
anterior_old_dir = spatial_section_dir / "old_anterior_section_example_1"
posterior_young_dir = spatial_section_dir / "young_posterior_section_example_1"
posterior_old_dir = spatial_section_dir / "old_posterior_section_example_1"

NEURONAL_LABELS = ["CNU GABA", "CTX GABA", "CTX Glut", "HY/AMY/TH"]
RANDOM_SEED = 42

if ALDEX_RESULT_MODE not in {"published", "recompute"}:
    raise ValueError("ALDEX_RESULT_MODE must be 'published' or 'recompute'.")
if ALDEX_RESULT_MODE == "published":
    required_mode_inputs = [config.additional_data_dir / "Additional_File_4.xlsx"]
    print("Published mode: using frozen pooled primary ALDEx results.")
else:
    required_mode_inputs = [aldex_results_file, gene_pivot_file]
    print(
        "Recompute mode: this downstream notebook consumes postprocessed "
        "outputs from the notebook 03/R workflow; it does not create them. "
        "See RECOMPUTE_WORKFLOW.md."
    )
missing_mode_inputs = [path for path in required_mode_inputs if not path.is_file()]
if missing_mode_inputs:
    raise FileNotFoundError(
        f"{ALDEX_RESULT_MODE.title()} mode is missing required ALDEx input(s): "
        + ", ".join(map(str, missing_mode_inputs))
    )

## 1. Recurrent-gene dot plot (Figure 3B)

The top 20 recurrent genes are taken from the signed cell type × region pivot table, with `Vwf` added as an additional vascular example. Dot color represents the ALDEx age effect estimate and dot size represents adjusted significance.


In [ ]:
def load_aldex_results(workbook):
    """Load one postprocessed ALDEx result table per workbook sheet."""
    xls = pd.ExcelFile(workbook)
    return {
        sheet: pd.read_excel(workbook, sheet_name=sheet)
        for sheet in xls.sheet_names
    }


def build_top_gene_scores(sig_results, pivot_file, n_top=20, extra_genes=("Vwf",)):
    """Create the long-form gene × cell type-region table used for Figure 3B."""
    pivot_df = (
        pivot_file.copy()
        if isinstance(pivot_file, pd.DataFrame)
        else pd.read_excel(pivot_file, index_col=0)
    )
    pivot_df.index.name = None

    top_genes = pivot_df.index[:n_top].tolist()
    top_genes.extend(g for g in extra_genes if g not in top_genes)

    rows = []
    for gene in top_genes:
        for ct_region, result_df in sig_results.items():
            gene_column = "gene" if "gene" in result_df.columns else "Gene"
            matches = result_df.loc[result_df[gene_column] == gene]
            if matches.empty:
                continue

            hit = matches.iloc[0]
            rows.append({
                "gene": gene,
                "celltype x region": ct_region.replace("_", " ").title(),
                "effect size": hit["age_binary:est"],
                "p value": hit["age_binary:pval.adj"],
            })

    scores = pd.DataFrame(rows)
    scores["p value"] = pd.to_numeric(scores["p value"], errors="coerce")
    scores["size"] = -np.log10(scores["p value"].clip(lower=1e-300))
    scores["celltype x region"] = pd.Categorical(
        scores["celltype x region"],
        categories=sorted(scores["celltype x region"].unique()),
        ordered=True,
    )
    return scores


if ALDEX_RESULT_MODE == "published":
    sig_results = load_pooled_strata(
        config.additional_data_dir / "Additional_File_4.xlsx",
        sheet_name="Celltype_region_postprocessed",
        stratum_column="Celltype x region",
        gene_column="Gene",
    )
    gene_pivot = pd.read_excel(
        config.additional_data_dir / "Additional_File_4.xlsx",
        sheet_name="Gene_Pivot_Table",
        index_col=0,
    )
elif ALDEX_RESULT_MODE == "recompute":
    sig_results = load_aldex_results(aldex_results_file)
    gene_pivot = gene_pivot_file
else:
    raise ValueError("ALDEX_RESULT_MODE must be 'published' or 'recompute'.")

top_gene_scores = build_top_gene_scores(
    sig_results,
    gene_pivot,
    n_top=20,
    extra_genes=("Vwf",),
)

top_gene_scores.head()


In [ ]:
def plot_recurrent_gene_dotplot(scores, p_legend=(0.05, 1e-4, 1e-8, 1e-16, 1e-24)):
    """Plot Figure 3B with a manually interpretable adjusted-p-value size legend."""
    fig, ax = plt.subplots(figsize=(10, 6))

    sns.scatterplot(
        data=scores,
        x="celltype x region",
        y="gene",
        hue="effect size",
        size="size",
        palette="RdBu_r",
        sizes=(30, 300),
        edgecolor="black",
        ax=ax,
    )

    size_values = scores["size"]
    vmin, vmax = size_values.min(), size_values.max()

    def p_to_size(p, smin=20, smax=200):
        x = -np.log10(p)
        if vmax == vmin:
            return (smin + smax) / 2
        return smin + (x - vmin) / (vmax - vmin) * (smax - smin)

    size_handles = [
        Line2D(
            [], [],
            marker="o",
            linestyle="",
            markersize=np.sqrt(max(p_to_size(p), 1)),
            markerfacecolor="gray",
            markeredgecolor="black",
            label=f"<{p:g}",
        )
        for p in p_legend
    ]

    ax.legend(
        handles=size_handles,
        title="Adjusted p value",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False,
    )
    ax.tick_params(axis="x", rotation=45)
    for label in ax.get_xticklabels():
        label.set_horizontalalignment("right")

    fig.tight_layout()
    return fig, ax


fig3b, ax3b = plot_recurrent_gene_dotplot(top_gene_scores)
plt.show()


## 2. Shared spatial-data helpers

These functions preserve the original workflow used to:

1. load each representative Vizgen section,
2. retain cells present in the processed MERFISH AnnData,
3. assign the manuscript broad-region and cell type × region labels,
4. permute age labels within batch and cell type × region strata, and
5. collapse neuronal subclasses to a single display label where required.


In [ ]:
def read_vizgen_section(path, library_id):
    """Read one resegmented Vizgen section."""
    return sq.read.vizgen(
        path=str(path),
        counts_file="cell_by_gene.csv",
        meta_file="cell_metadata.csv",
        transformation_file="micron_to_mosaic_pixel_transform.csv",
        library_id=library_id,
    )


def subset_spatial_to_sample(spatial_adata, processed_adata, sample_id):
    """Retain the processed cells belonging to one sample and preserve their order."""
    sample_ids = processed_adata.obs_names[processed_adata.obs["sample"] == sample_id]
    return spatial_adata[spatial_adata.obs_names.isin(sample_ids)].copy()


def collapse_neuronal_labels(adata, column="cell_type"):
    """Collapse neuronal subclasses into the display label 'Neuronal'."""
    adata.obs[column] = adata.obs[column].astype(str)
    adata.obs.loc[adata.obs[column].isin(NEURONAL_LABELS), column] = "Neuronal"
    adata.obs[column] = pd.Categorical(adata.obs[column])


def transfer_obs_columns(spatial_adata, processed_adata, sample_id, columns):
    """Transfer processed metadata to a spatial sample by cell identifier."""
    source = processed_adata.obs.loc[
        processed_adata.obs["sample"] == sample_id,
        columns,
    ]
    for column in columns:
        spatial_adata.obs[column] = source[column].reindex(spatial_adata.obs_names)


def permute_age_within_batch_and_stratum(
    adata,
    stratum_col="ct_region",
    batch_col="batch",
    age_col="Age",
    seed=RANDOM_SEED,
):
    """Reproduce the original within-batch age permutation for each stratum."""
    permuted = pd.Series(index=adata.obs_names, dtype=object, name="Age_perm")

    for stratum in adata.obs[stratum_col].cat.categories:
        obs = adata.obs.loc[adata.obs[stratum_col] == stratum].copy()
        rng = np.random.default_rng(seed=seed)

        for batch in obs[batch_col].cat.categories:
            mask = obs[batch_col] == batch
            obs.loc[mask, "Age_perm"] = rng.permutation(
                obs.loc[mask, age_col].values
            )

        permuted.loc[obs.index] = obs["Age_perm"]

    adata.obs["Age_perm"] = pd.Categorical(permuted)


In [ ]:
def assign_analysis_regions(adata, ontology, include_thal_hyp=False):
    """Assign the broad-region and cell type × region labels used in the analysis."""
    target_structures = [
        ["Isocortex"],
        ["Hippocampal formation"],
        ["Cortical subplate"],
        ["Olfactory areas"],
        ["Cerebral nuclei"],
        ["fiber tracts"],
        ["ventricular systems"],
    ]

    if include_thal_hyp:
        target_structures[1:1] = [["Thalamus"]]
        target_structures.insert(2, ["Hypothalamus"])

    region_adatas = {
        structure[0].replace(" ", "_"): None
        for structure in target_structures
    }

    for structure in target_structures:
        key = structure[0].replace(" ", "_")
        target_ids = ontology.loc[ontology["name"].isin(structure), "id"].astype(str)
        target_paths = [f"/{x}/" for x in target_ids]

        sub_ontology = ontology[
            ontology["id"].isin(
                adata.obs["Region"].fillna(0).astype(int).tolist()
            )
        ]

        region_ids = sub_ontology.loc[
            sub_ontology["structure_id_path"].isin([
                path
                for path in sub_ontology["structure_id_path"].tolist()
                if any(
                    target in path or path in target
                    for target in target_paths
                )
            ]),
            "id",
        ].tolist()

        region_adatas[key] = adata[adata.obs["Region"].isin(region_ids)].copy()

    region_adatas["Limbic_Olf"] = pp.merge_adatas(
        [
            region_adatas["Hippocampal_formation"],
            region_adatas["Cortical_subplate"],
            region_adatas["Olfactory_areas"],
        ],
        replicate=True,
        section=True,
        exp_cond="Age",
        layers="raw_counts",
    )
    region_adatas["Non_Gray"] = pp.merge_adatas(
        [
            region_adatas["ventricular_systems"],
            region_adatas["fiber_tracts"],
        ],
        replicate=True,
        section=True,
        exp_cond="Age",
        layers="raw_counts",
    )

    analysis_regions = {
        "Isocortex": region_adatas["Isocortex"],
        "cerebralnuclei": region_adatas["Cerebral_nuclei"].copy(),
        "limbicolf": region_adatas["Limbic_Olf"].copy(),
        "nongray": region_adatas["Non_Gray"].copy(),
    }

    region_labels = {
        "Isocortex": "Isocortex",
        "cerebralnuclei": "Cerebral Nuclei",
        "limbicolf": "Limbic/Olfactory",
        "nongray": "Non-Gray Matter",
    }

    if include_thal_hyp:
        region_adatas["Thal_Hyp"] = pp.merge_adatas(
            [region_adatas["Thalamus"], region_adatas["Hypothalamus"]],
            replicate=True,
            section=True,
            exp_cond="Age",
            layers="raw_counts",
        )
        analysis_regions["thalhyp"] = region_adatas["Thal_Hyp"].copy()
        region_labels["thalhyp"] = "Thalamus/Hypothalamus"

    ct_region_parts = []
    broad_region_parts = []

    for region_key, region_adata in analysis_regions.items():
        region_adata.obs["ct_region"] = (
            region_key + "_" + region_adata.obs["cell_type"].astype(str).str.lower()
        )
        region_adata.obs["ct_region"] = pd.Categorical(
            region_adata.obs["ct_region"]
        )

        region_adata.obs["broad_region"] = pd.Categorical(
            [region_labels[region_key]] * region_adata.n_obs
        )

        ct_region_parts.append(region_adata.obs[["ct_region"]])
        broad_region_parts.append(region_adata.obs[["broad_region"]])

    ct_region_df = pd.concat(ct_region_parts).reindex(adata.obs_names)
    broad_region_df = pd.concat(broad_region_parts).reindex(adata.obs_names)

    adata.obs["ct_region"] = pd.Categorical(
        ct_region_df["ct_region"].fillna("Not in subset")
    )
    adata.obs["broad_region"] = pd.Categorical(
        broad_region_df["broad_region"].fillna("Not in subset")
    )

    return analysis_regions


## 3. Representative anterior sections

The anterior pair corresponds to samples 1 (old) and 2 (young). The anterior plane does not contain all anatomical groups used in the full analysis, so thalamus/hypothalamus is intentionally excluded here, matching the original notebook.


In [ ]:
ontology = pd.read_excel(ontology_file)
adata = sc.read_h5ad(adata_file)

aepd_young = read_vizgen_section(anterior_young_dir, library_id="Yng_1")
aepd_old = read_vizgen_section(anterior_old_dir, library_id="Old_1")

aoss = subset_spatial_to_sample(aepd_old, adata, sample_id=1)
ayss = subset_spatial_to_sample(aepd_young, adata, sample_id=2)

# Preserve the original region-assignment logic before collapsing neuronal
# labels solely for display in these spatial plots.
anterior_regions = assign_analysis_regions(
    adata,
    ontology,
    include_thal_hyp=False,
)
permute_age_within_batch_and_stratum(adata)

transfer_obs_columns(
    aoss, adata, sample_id=1,
    columns=["cell_type", "broad_region", "ct_region", "Age_perm", "Age"],
)
transfer_obs_columns(
    ayss, adata, sample_id=2,
    columns=["cell_type", "broad_region", "ct_region", "Age_perm", "Age"],
)

collapse_neuronal_labels(aoss)
collapse_neuronal_labels(ayss)

iso_astro_ao = aoss[aoss.obs["ct_region"] == "Isocortex_astro"].copy()
iso_astro_ay = ayss[ayss.obs["ct_region"] == "Isocortex_astro"].copy()

# Preserve the original young-age display color.
iso_astro_ay.uns["Age_colors"] = ["#ff7f0e"]


### Figure 2A: anterior within-batch permutation example


In [ ]:
sq.pl.spatial_scatter(
    iso_astro_ao,
    shape=None,
    color=["Age", "Age_perm"],
)

sq.pl.spatial_scatter(
    iso_astro_ay,
    shape=None,
    color=["Age", "Age_perm"],
)


### Figure 3A: anterior cell-type and broad-region maps


In [ ]:
# Representative young hemisphere: cell types
sq.pl.spatial_scatter(
    ayss,
    shape=None,
    color="cell_type",
)

# Matched old hemisphere: broad regions
sq.pl.spatial_scatter(
    aoss,
    shape=None,
    color="broad_region",
)


### Figure 6C: anterior spatial expression of `Cpe`

The same 1st–99th percentile expression scale, calculated from the old section as in the original notebook, is applied to both sections.


In [ ]:
cpe_values = aoss[:, "Cpe"].X
cpe_values = (
    cpe_values.toarray().ravel()
    if hasattr(cpe_values, "toarray")
    else np.asarray(cpe_values).ravel()
)
cpe_vmin, cpe_vmax = np.percentile(cpe_values, [1, 99])

sq.pl.spatial_scatter(
    aoss,
    shape=None,
    color="Cpe",
    cmap="plasma",
    vmin=cpe_vmin,
    vmax=cpe_vmax,
)

sq.pl.spatial_scatter(
    ayss,
    shape=None,
    color="Cpe",
    cmap="plasma",
    vmin=cpe_vmin,
    vmax=cpe_vmax,
)


## 4. Representative posterior sections

The posterior pair corresponds to samples 11 (old) and 12 (young). The processed AnnData is deliberately reloaded before this block, matching the original notebook. Unlike the anterior block, the posterior region assignment includes thalamus/hypothalamus.

This block also reproduces the original export of the non-neuronal AnnData used in downstream ALDEx analyses.


In [ ]:
# Reload the processed AnnData before posterior-region preparation.
adata = sc.read_h5ad(adata_file)

pvh_young = read_vizgen_section(posterior_young_dir, library_id="Yng_2")
pvh_old = read_vizgen_section(posterior_old_dir, library_id="Old_2")

poss = subset_spatial_to_sample(pvh_old, adata, sample_id=11)
pyss = subset_spatial_to_sample(pvh_young, adata, sample_id=12)

# Preserve the original ordering: collapse neuronal labels in the processed
# AnnData before constructing posterior ct_region labels.
collapse_neuronal_labels(adata)

posterior_regions = assign_analysis_regions(
    adata,
    ontology,
    include_thal_hyp=True,
)

# The deposited glial AnnData was generated from the ct_temp non-neuronal
# subset during preprocessing. Figure reproduction intentionally does not
# regenerate or overwrite that deposited input here.

permute_age_within_batch_and_stratum(adata)

transfer_obs_columns(
    poss, adata, sample_id=11,
    columns=["cell_type", "broad_region", "ct_region", "Age_perm", "Age"],
)
transfer_obs_columns(
    pyss, adata, sample_id=12,
    columns=["cell_type", "broad_region", "ct_region", "Age_perm", "Age"],
)

collapse_neuronal_labels(poss)
collapse_neuronal_labels(pyss)

iso_astro_po = poss[poss.obs["ct_region"] == "Isocortex_astro"].copy()
iso_astro_py = pyss[pyss.obs["ct_region"] == "Isocortex_astro"].copy()
iso_astro_py.uns["Age_colors"] = ["#ff7f0e"]


### Figure 2A: posterior within-batch permutation example


In [ ]:
sq.pl.spatial_scatter(
    iso_astro_po,
    shape=None,
    color=["Age", "Age_perm"],
)

sq.pl.spatial_scatter(
    iso_astro_py,
    shape=None,
    color=["Age", "Age_perm"],
)


### Figure 3A: posterior broad-region and cell-type maps


In [ ]:
# Representative old hemisphere: broad regions
sq.pl.spatial_scatter(
    poss,
    shape=None,
    color="broad_region",
)

# Matched young hemisphere: cell types
sq.pl.spatial_scatter(
    pyss,
    shape=None,
    color="cell_type",
)


## 5. Raw-expression split violins (Figure 3C)

For a selected cell type and gene, the plot shows raw-count distributions across cell type × region strata, split by age. Violin shading reflects the overall percentage of cells expressing the gene within each age group, matching the original plotting logic.


In [ ]:
def split_violin_ct_region(
    adata_ss,
    cell_type="EC",
    gene="Vwf",
    this_obs="Age",
    layer="raw_counts",
):
    """Plot age-split raw-expression violins across cell type × region strata."""
    adata_ss = adata_ss[adata_ss.obs["cell_type"] == cell_type].copy()
    adata_ss = adata_ss[
        adata_ss.obs["ct_region"] != "Not in subset"
    ].copy()

    if layer == "raw_counts":
        expression = adata_ss[:, gene].layers["raw_counts"]
    else:
        expression = adata_ss[:, gene].X

    if hasattr(expression, "A"):
        expression = expression.A.ravel()
    elif hasattr(expression, "toarray"):
        expression = expression.toarray().ravel()
    else:
        expression = np.asarray(expression).ravel()

    df = pd.DataFrame({
        "expression": expression,
        this_obs: adata_ss.obs[this_obs].values,
        "ct_region": adata_ss.obs["ct_region"].values,
    })
    df["gene"] = gene
    df["expressed"] = df["expression"] > 0
    df["gene_group"] = (
        df["gene"].astype(str) + "_" + df[this_obs].astype(str)
    )

    percent_expressing = (
        df.groupby(["gene", this_obs], observed=False)["expressed"]
        .mean()
        .mul(100)
        .reset_index(name="percent_expressing")
    )
    percent_expressing["gene_group"] = (
        percent_expressing["gene"].astype(str)
        + "_"
        + percent_expressing[this_obs].astype(str)
    )
    percent_expressing["expression_bin"] = pd.cut(
        percent_expressing["percent_expressing"],
        bins=list(range(0, 110, 10)),
        right=False,
    )

    group_names = list(df[this_obs].dropna().unique())
    base_palette = sns.color_palette("Set2", n_colors=len(group_names))
    group_base_colors = dict(zip(group_names, base_palette))

    binned_shades = {
        (gene, group): sns.light_palette(
            group_base_colors[group],
            n_colors=10,
        )
        for group in group_names
    }

    def get_binned_color(row):
        if pd.isna(row["expression_bin"]):
            return "gray"
        bin_index = int(row["expression_bin"].left // 10)
        return binned_shades[(row["gene"], row[this_obs])][bin_index]

    percent_expressing["color"] = percent_expressing.apply(
        get_binned_color,
        axis=1,
    )

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.violinplot(
        data=df,
        x="ct_region",
        y="expression",
        hue=this_obs,
        split=True,
        density_norm="width",
        inner=None,
        palette=base_palette,
        ax=ax,
    )

    for violin, (_, row) in zip(
        ax.findobj(PolyCollection),
        percent_expressing.iterrows(),
    ):
        violin.set_facecolor(to_rgb(row["color"]))

    ax.grid(True)
    ax.tick_params(axis="x", rotation=45)
    for label in ax.get_xticklabels():
        label.set_horizontalalignment("right")
    ax.set_ylim(-3, 40)

    fig.tight_layout()
    return fig, ax


In [ ]:
# Figure 3C, top panel
fig3c_top, ax3c_top = split_violin_ct_region(
    adata,
    cell_type="EC",
    gene="Vwf",
)
plt.show()

# Figure 3C, bottom panel
fig3c_bottom, ax3c_bottom = split_violin_ct_region(
    adata,
    cell_type="VSMC",
    gene="Col1a1",
)
plt.show()
